# Problema della dieta
## Obiettivo: 
Minimizzare il costo totale della dieta
## Variabili: 
Quantità di ogni alimento da acquistare
## Vincoli: 
Soddisfare i fabbisogni nutrizionali

## Modello matematico:
$$
\min \sum_{j=1}^n c_j x_j 
$$
$$
\text{S.t.} \sum_{j=1}^n a_{ij} x_j \geq b_i \quad \forall i=1,...,m 
$$
$$
x_j \geq 0
$$

## Dati

### 1. Costo di ogni alimento
$$c_j$$

<table>
  <tr>
    <th>pane</th>
    <th>lenticchie</th>
    <th>carne</th>
    <th>verdura</th>
  </tr>
  <tr>
    <td>3</td>
    <td>2</td>
    <td>15</td>
    <td>1</td>
  </tr>
</table>

### 2. Quantità di ogni nutriente richiesta
$$b_i$$

<table>
  <tr>
    <th>carboidrati</th>
    <th>proteine</th>
    <th>grassi</th>
  </tr>
  <tr>
    <td>15</td>
    <td>6</td>
    <td>3</td>
  </tr>
</table>

### 3. Quantità di ogni nutriente presente in ogni alimento
$$a_{ij}$$

<table>
  <tr>
    <th>nutriente</th>
    <th>pane</th>
    <th>lenticchie</th>
    <th>carne</th>
    <th>verdura</th>
  </tr>
  <tr>
    <td>carboidrati</td>
    <td>7</td>
    <td>3</td>
    <td>1</td>
    <td>2</td>
  </tr>
  <tr>
    <td>proteine</td>
    <td>2</td>
    <td>6</td>
    <td>10</td>
    <td>1</td>
  </tr>
  <tr>
    <td>grassi</td>
    <td>0</td>
    <td>1</td>
    <td>3</td>
    <td>0</td>
  </tr>
</table>


In [1]:
import pyomo.environ as pyo

alimenti = ['pane', 'lenticchie', 'carne', 'verdura']
costi = [3, 2, 15, 1]
c = dict(zip(alimenti, costi))

nutrienti = ['carboidrati', 'proteine', 'grassi']
richiesta = dict(zip(nutrienti, [15, 6, 3]))

mix_carboidrati = [7, 3, 1, 2]
mix_proteine = [2, 6, 10, 1]
mix_grassi = [0, 1, 3, 0]
mix_uso = dict(zip(nutrienti, [mix_carboidrati, mix_proteine, mix_grassi]))

model = pyo.ConcreteModel(name="Problema della dieta")

# Variabile decisionale: quantita acquistata di ogni alimento
model.x = pyo.Var(alimenti, within=pyo.NonNegativeIntegers)

def funzione_obiettivo(m: pyo.ConcreteModel) -> pyo.Expression:
    # Minimizza il costo totale della dieta
    return pyo.quicksum(c[alimento] * m.x[alimento] for alimento in alimenti)

model.obiettivo = pyo.Objective(rule=funzione_obiettivo, sense=pyo.minimize)

def vincoli_modello(m: pyo.ConcreteModel, nutriente: str) -> pyo.Expression:
    """
    Calcola la quantita totale assunta del nutriente considerato.
    """
    quantita_assunta = 0.0
    for j in range(len(alimenti)):
        nome_alimento = alimenti[j]
        quantita_nutriente = mix_uso[nutriente][j]
        quantita_assunta += quantita_nutriente * m.x[nome_alimento]
    print(quantita_assunta)
    return quantita_assunta >= richiesta[nutriente]

model.vincoli = pyo.Constraint(nutrienti, rule=vincoli_modello)

ottimizzatore = pyo.SolverFactory('glpk').solve(model, tee=False)

model.display()

7*x[pane] + 3*x[lenticchie] + x[carne] + 2*x[verdura]
2*x[pane] + 6*x[lenticchie] + 10*x[carne] + x[verdura]
0*x[pane] + x[lenticchie] + 3*x[carne] + 0*x[verdura]
Model Problema della dieta

  Variables:
    x : Size=4, Index={pane, lenticchie, carne, verdura}
        Key        : Lower : Value : Upper : Fixed : Stale : Domain
             carne :     0 :   0.0 :  None : False : False : NonNegativeIntegers
        lenticchie :     0 :   3.0 :  None : False : False : NonNegativeIntegers
              pane :     0 :   1.0 :  None : False : False : NonNegativeIntegers
           verdura :     0 :   0.0 :  None : False : False : NonNegativeIntegers

  Objectives:
    obiettivo : Size=1, Index=None, Active=True
        Key  : Active : Value
        None :   True :   9.0

  Constraints:
    vincoli : Size=3
        Key         : Lower : Body : Upper
        carboidrati :  15.0 : 16.0 :  None
           proteine :   6.0 : 20.0 :  None
             grassi :   3.0 :  3.0 :  None
